# 02 — Modelos SOTA, Métricas Clínicas & Interpretabilidade

**Branch 2 — Laura** | Continuação do pipeline iniciado em `01_eda_dados.ipynb`

---

**Pré-requisito:** Ter corrido o notebook `01_eda_dados.ipynb` para que os ficheiros `data/MIQR-CC-Dataset/splits/train.csv`, `val.csv` e `test.csv` existam.

## 0. Setup do Ambiente (Colab)

> Corre esta célula **apenas** se estiveres no Google Colab.

In [ ]:
# !pip install wandb grad-cam -qU
# import wandb
# wandb.login()

## 1. Imports e Configuração

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Adicionar a pasta src ao path para importar os módulos
sys.path.append(os.path.join("..", "src"))

from data_setup import MIQRDataset, train_transforms, val_transforms
from models import get_model
from train_wandb import TrainerWandb
from metrics import MetricsEvaluator
from gradcam_utils import InterpretabilityTools, get_target_layers_for_model
from noise_robustness import evaluate_robustness, print_robustness_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device em uso:', device)

# Caminhos — relativos à pasta notebooks/
DATA_DIR = os.path.join("..", "data", "MIQR-CC-Dataset")
TRAIN_CSV = os.path.join(DATA_DIR, "splits", "train.csv")
VAL_CSV   = os.path.join(DATA_DIR, "splits", "val.csv")
TEST_CSV  = os.path.join(DATA_DIR, "splits", "test.csv")

CLASS_NAMES = ['Biliary_Leaks', 'Lithiasis', 'Stricture', 'Normal']
# Pesos calculados no notebook 01 para lidar com o desequilíbrio
CLASS_WEIGHTS = torch.tensor([2.5960, 0.5399, 1.0000, 1.3110], dtype=torch.float32).to(device)
print("Class weights:", CLASS_WEIGHTS)

## 2. DataLoaders

Usa a classe `MIQRDataset` da tua colega (`src/data_setup.py`) que já aplica **CLAHE**, **Data Augmentation** e **Normalização ImageNet**.

In [ ]:
BATCH_SIZE = 32

dataset_treino = MIQRDataset(TRAIN_CSV, DATA_DIR, transform=train_transforms)
dataset_val    = MIQRDataset(VAL_CSV,   DATA_DIR, transform=val_transforms)
dataset_teste  = MIQRDataset(TEST_CSV,  DATA_DIR, transform=val_transforms)

train_loader = DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(dataset_val,    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(dataset_teste,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Treino:    {len(dataset_treino)} imagens | {len(train_loader)} batches")
print(f"Validação: {len(dataset_val)} imagens | {len(val_loader)} batches")
print(f"Teste:     {len(dataset_teste)} imagens | {len(test_loader)} batches")

## 3. Configuração do Modelo SOTA e Treino com WandB

Troca `model_name` para comparar os diferentes modelos:
- `'densenet121'` —  **Recomendado para raio-X** (arquitetura do CheXNet, Stanford)
- `'efficientnet_v2_s'` —  Rápido e eficiente
- `'resnet50'` —  Mais potente que a ResNet-18 da baseline
- `'vit_b_16'` —  Vision Transformer (gera Attention Maps)

In [ ]:
config = {
    'epochs': 50,
    'batch_size': BATCH_SIZE,
    'learning_rate': 1e-4,
    'model_name': 'densenet121',
    'transfer_learning': False,   # False = fine-tuning completo
    'class_weights': True         # Usar os pesos calculados no notebook 01
}

criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
model = get_model(config['model_name'], num_classes=4, feature_extracting=config['transfer_learning'])
optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
print(f"Modelo: {config['model_name']} | Fine-tuning: {not config['transfer_learning']}")

trainer = TrainerWandb(model, train_loader, val_loader, criterion, optimizer, device, CLASS_NAMES)
best_f1 = trainer.train_and_evaluate(config, epochs=config['epochs'])
print(f"\nMelhor F1-Score Macro: {best_f1:.4f}  (baseline a bater: 0.738)")

## 4. Avaliação Final no Conjunto de Teste

Métricas obrigatórias: **F1-Score Macro**, **Matriz de Confusão**, **Curvas AUC-ROC**.

In [ ]:
evaluator = MetricsEvaluator(CLASS_NAMES)

# Carregar o melhor modelo gravado pelo trainer
model.load_state_dict(torch.load('checkpoints/best_model.pth', map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        probs = torch.softmax(model(inputs.to(device)), dim=1)
        all_preds.append(probs.cpu())
        all_labels.append(labels)

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

results = evaluator.evaluate(all_labels, all_preds)
print(f"\n=== Resultados no Conjunto de Teste ===")
print(f"F1-Score Macro: {results['f1_macro']:.4f}  (baseline: 0.738)")
print(f"AUC-ROC:        {results['auc_roc']:.4f}")
print("\n" + results['report'])

# Matriz de Confusão
evaluator.plot_confusion_matrix(
    results['y_true'], results['y_pred_labels'],
    title=f"Matriz de Confusão — {config['model_name']}",
    save_path=f"../outputs/confusion_matrix_{config['model_name']}.png"
)

# Curvas AUC-ROC
evaluator.plot_roc_curves(
    all_labels, all_preds,
    title=f"Curvas AUC-ROC — {config['model_name']}",
    save_path=f"../outputs/roc_curves_{config['model_name']}.png"
)

## 5. Teste de Robustez ao Ruído

Avalia o modelo sob variações comuns em imagens de fluoroscopia:
ruído Gaussiano, variações de contraste e artefactos de pixels.

In [ ]:
robustness_results = evaluate_robustness(
    model=model,
    val_loader=val_loader,
    evaluator=evaluator,
    device=device
)
print_robustness_report(robustness_results)

## 6. Interpretabilidade: Grad-CAM

Requisito **obrigatório** do professor. Gera heatmaps que mostram onde o modelo está a focar.

> ✅ Verifica se o foco está nos **ductos biliares** e não em artefactos externos.

In [ ]:
os.makedirs("../outputs/gradcam", exist_ok=True)

target_layers = get_target_layers_for_model(model, config['model_name'])
interpreter = InterpretabilityTools(model, target_layers)

# Gerar Grad-CAM para os primeiros 4 exemplos do conjunto de teste
inputs, labels = next(iter(test_loader))

fig, axes = plt.subplots(4, 2, figsize=(10, 18))
fig.suptitle(f'Grad-CAM — {config["model_name"]}', fontsize=14)

for i in range(4):
    img_tensor = inputs[i:i+1].to(device)
    img_rgb = inputs[i].permute(1, 2, 0).cpu().numpy()
    img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() - img_rgb.min() + 1e-8)
    img_rgb = img_rgb.astype(np.float32)

    classe_verdadeira = labels[i].item()
    heatmap_img = interpreter.generate_gradcam(
        img_tensor, img_rgb,
        target_category=classe_verdadeira,
        save_path=f"../outputs/gradcam/gradcam_{i}_{CLASS_NAMES[classe_verdadeira]}.png"
    )

    axes[i][0].imshow(img_rgb, cmap='gray')
    axes[i][0].set_title(f'Original: {CLASS_NAMES[classe_verdadeira]}')
    axes[i][0].axis('off')

    axes[i][1].imshow(heatmap_img)
    axes[i][1].set_title('Grad-CAM (zona de atenção)')
    axes[i][1].axis('off')

plt.tight_layout()
plt.savefig("../outputs/gradcam/gradcam_panel.png", dpi=150)
plt.show()
print("Heatmaps guardados em outputs/gradcam/")